# 03 — Model Comparison: Baseline Models

**Phase 5 — Baseline Modeling**

**Objective**: Evaluate baseline predictive models (Naive and Linear Regression)
and establish a performance floor for the bike availability forecasting task.

**Models compared**:
1. **Naive Baseline** — predicts `bikes_lag_1` (last known value)
2. **Linear Regression** — OLS on all 15 features

**Metrics**: MAE, RMSE, R²

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.dataset.features import FEATURE_COLS, TARGET_COL
from src.model.evaluate import compute_metrics, per_hour_metrics, per_station_metrics

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"

DATA_DIR = Path.cwd().parent / "data" / "processed"

print("Setup complete ✓")

## 2. Load Data and Models

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")

naive = joblib.load(DATA_DIR / "naive.joblib")
lr = joblib.load(DATA_DIR / "lr.joblib")

print(f"Train: {len(train_df):,} rows")
print(f"Test:  {len(test_df):,} rows")
print(f"Features: {len(FEATURE_COLS)}")

## 3. Generate Predictions

In [ ]:
test_df = test_df.copy()
test_df["y_pred_naive"] = naive.predict(test_df[FEATURE_COLS])
test_df["y_pred_lr"] = lr.predict(test_df[FEATURE_COLS])

print("Predictions generated ✓")

## 4. Metrics Summary

In [ ]:
y_true = test_df[TARGET_COL].values

metrics_naive = compute_metrics(y_true, test_df["y_pred_naive"].values)
metrics_lr = compute_metrics(y_true, test_df["y_pred_lr"].values)

summary = pd.DataFrame(
    {"Naive Baseline": metrics_naive, "Linear Regression": metrics_lr}
).T
summary.index.name = "Model"

print("=" * 55)
print("MODEL COMPARISON")
print("=" * 55)
print(summary.to_string(float_format="{:.4f}".format))

if metrics_lr["mae"] < metrics_naive["mae"]:
    improvement = (1 - metrics_lr["mae"] / metrics_naive["mae"]) * 100
    print(f"\nLinear Regression improves MAE by {improvement:.1f}% over Naive Baseline")
else:
    print("\nNote: Linear Regression did NOT improve over Naive Baseline")

## 5. Actual vs Predicted

In [ ]:
sample = test_df.sample(min(5000, len(test_df)), random_state=42)
max_val = max(
    sample[TARGET_COL].max(), sample["y_pred_naive"].max(), sample["y_pred_lr"].max()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pred_col, title in [
    (axes[0], "y_pred_naive", "Naive Baseline"),
    (axes[1], "y_pred_lr", "Linear Regression"),
]:
    ax.scatter(sample[TARGET_COL], sample[pred_col], alpha=0.15, s=5, color="tab:blue")
    ax.plot([0, max_val], [0, max_val], "r--", alpha=0.5, label="Perfect prediction")
    ax.set_xlabel("Actual (y)")
    ax.set_ylabel("Predicted")
    ax.set_title(title)
    ax.legend()
    ax.set_aspect("equal", adjustable="box")

plt.suptitle("Actual vs Predicted — Test Set", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred_col, title, color in [
    (axes[0], "y_pred_naive", "Naive Baseline", "tab:orange"),
    (axes[1], "y_pred_lr", "Linear Regression", "tab:blue"),
]:
    error = test_df[TARGET_COL] - test_df[pred_col]
    ax.hist(error, bins=60, color=color, edgecolor="white", alpha=0.8)
    ax.axvline(
        error.mean(), color="red", linestyle="--", label=f"Mean: {error.mean():.2f}"
    )
    ax.axvline(0, color="black", linestyle="-", alpha=0.3)
    ax.set_xlabel("Error (actual - predicted)")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend()

plt.suptitle("Error Distribution — Test Set", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Per-Hour MAE

In [ ]:
hour_naive = per_hour_metrics(test_df, TARGET_COL, "y_pred_naive")
hour_lr = per_hour_metrics(test_df, TARGET_COL, "y_pred_lr")

fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(hour_naive))
width = 0.35

ax.bar(
    x - width / 2,
    hour_naive["mae"],
    width,
    label="Naive",
    color="tab:orange",
    alpha=0.8,
)
ax.bar(
    x + width / 2,
    hour_lr["mae"],
    width,
    label="Linear Regression",
    color="tab:blue",
    alpha=0.8,
)

ax.set_xlabel("Hour of Day")
ax.set_ylabel("MAE")
ax.set_title("Mean Absolute Error by Hour")
ax.set_xticks(x)
ax.set_xticklabels(hour_naive.index)
ax.legend()
plt.tight_layout()
plt.show()

## 8. Per-Station MAE Analysis

In [ ]:
station_lr = per_station_metrics(test_df, TARGET_COL, "y_pred_lr")

# Top 10 hardest stations
top10 = station_lr.nlargest(10, "mae")
bottom10 = station_lr.nsmallest(10, "mae")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top10.index.astype(str), top10["mae"], color="tab:red", alpha=0.8)
axes[0].set_xlabel("MAE")
axes[0].set_title("Top 10 Hardest Stations (highest MAE)")
axes[0].invert_yaxis()

axes[1].barh(bottom10.index.astype(str), bottom10["mae"], color="tab:green", alpha=0.8)
axes[1].set_xlabel("MAE")
axes[1].set_title("Top 10 Easiest Stations (lowest MAE)")
axes[1].invert_yaxis()

plt.suptitle("Per-Station MAE — Linear Regression", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nStation MAE distribution:")
print(station_lr["mae"].describe().to_string())

## 9. Conclusion

### Results

| Model | MAE | RMSE | R² |
|-------|-----|------|----|
| Naive Baseline | TBD | TBD | TBD |
| Linear Regression | TBD | TBD | TBD |

*(Fill in after running with real data)*

### Findings

1. **Naive baseline** provides a strong starting point — bike availability is highly autocorrelated
2. **Linear Regression** should improve over naive by leveraging temporal and rolling features
3. **Hardest hours** to predict are likely during peak commute times (high variance)
4. **Hardest stations** are those with high-traffic / high-variability patterns

### Next Steps

Phase 6 will introduce **LightGBM** with hyperparameter tuning to capture non-linear
patterns that Linear Regression cannot model.